# Fine-Tune `multilingual-e5-base` for Swiss Legal Retrieval

Domain-adapts the embedding model using MNRL (Multiple Negatives Ranking Loss) on the 1,139 labelled training examples.

**Pipeline:**
1. Build inverted citation graph: statute → decisions that cite it
2. Scan `court_considerations.csv` once to collect positive chunks
3. Build training triplets: (query, positive_chunk, cross-domain_hard_negative)
4. Fine-tune `intfloat/multilingual-e5-base` for 2 epochs
5. Save to `models/e5-base-swiss-legal-tuned/`
6. Re-index LanceDB with the new model (last section — run separately)

In [1]:
import json
import re
import random
import sys
from collections import defaultdict, Counter
from pathlib import Path
from sentence_transformers import InputExample
import gc, os
from sentence_transformers import SentenceTransformer
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer
from datasets import Dataset as HFDataset


import pandas as pd
from tqdm.auto import tqdm
import torch

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

REPO_ROOT  = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH  = REPO_ROOT / 'data'
INDEX_PATH = DATA_PATH / 'processed'
MODEL_OUT  = REPO_ROOT / 'models' / 'e5-base-swiss-legal-tuned'
MODEL_OUT.parent.mkdir(exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Repo   : {REPO_ROOT}')

Device : cuda
GPU    : NVIDIA GeForce RTX 4050 Laptop GPU
VRAM   : 6.4 GB
Repo   : c:\Users\david\Desktop\project_ssl\Omnilex-Agentic-Retrieval-Competition


C:\Users\david\AppData\Local\Temp\ipykernel_38256\3520663267.py:10: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import MultipleNegativesRankingLoss
C:\Users\david\AppData\Local\Temp\ipykernel_38256\3520663267.py:11: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import SentenceTransformerTrainingArguments
C:\Users\david\AppData\Local\Temp\ipykernel_38256\3520663267.py:12: DeprecationWarning: Importing from 'sentence_transformers.trainer' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.trainer' instead.
  from sentence_transformers.trainer impor

## 1. Load Training Data

In [2]:
train_df = pd.read_csv(DATA_PATH / 'train.csv')
print(f'Training examples : {len(train_df)}')
print(f'Columns           : {list(train_df.columns)}')

train_df['n_citations'] = train_df['gold_citations'].apply(
    lambda x: len(str(x).split(';'))
)
print(f'\nGold citations per query:')
print(train_df['n_citations'].describe().to_string())
print(f'\nSample queries:')
for _, row in train_df.head(3).iterrows():
    q = str(row['query'])[:120].replace('\n', ' ')
    c = str(row['gold_citations'])[:80]
    print(f'  [{row["query_id"]}] {q}...')
    print(f'           citations: {c}')
    print()

Training examples : 1139
Columns           : ['query_id', 'query', 'gold_citations']

Gold citations per query:
count    1139.000000
mean        4.090430
std         4.512206
min         1.000000
25%         1.000000
50%         2.000000
75%         5.000000
max        44.000000

Sample queries:
  [train_0001] Die A AG betreibt seit den 1970er-Jahren auf der Parzelle Nr. yyy (Wohn- und Gewerbezone) ein Recyclingunternehmen. Das ...
           citations: Art. 10a Abs. 1 USG;Art. 2 Abs. 1 UVPV;Art. 10a Abs. 1 UVG

  [train_0002] Die Alpha Consulting AG kann nun ihr Grundstück neu direkt an die soeben erstellte Kana lisationsleitung in der angrenze...
           citations: Art. 975 ZGB;Art. 974 Abs. 2 ZGB;Art. 973 ZGB;Art. 661 ZGB;Art. 956a ZGB;Art. 95

  [train_0003] Das Kompetenzzentrum Völkerstrafrecht bei der Bundesanwaltschaft erhält einen Tipp von syrischen Flüchtlingen. In einer ...
           citations: Art. 264m StGB



## 2. Build Inverted Citation Graph

`citation_graph.json` maps `decision_id → [statutes]`. We invert it to `statute → [decision_ids]` so we can look up which court decisions to use as positive examples for each gold statute.

In [3]:
# Citation graph no longer needed for training triplet construction.
# Positives now come directly from laws_de.csv via laws_lookup (section 3).
print("Section 2 skipped — citation graph not needed for direct laws-lookup approach.")

Section 2 skipped — citation graph not needed for direct laws-lookup approach.


## 3. Load Laws Lookup

Direct O(1) lookup of statute text from `laws_de.csv`. No 2.5M-row CSV scan needed.

- **Exact match**: `"Art. 221 Abs. 1 StPO"` → statutory text directly
- **Prefix fallback**: `"Art. 975 ZGB"` → matches `"Art. 975 Abs. 1 ZGB"` by stripping `Abs.`

This gives the most direct positive signal: the model learns to match a legal query to the actual statute text it cites, not an indirect court chunk.

In [4]:
import csv
import re as _re

# Load laws_de.csv — exact match dict
laws_lookup: dict[str, str] = {}
with open(DATA_PATH / 'laws_de.csv', encoding='utf-8', newline='') as f:
    for r in csv.DictReader(f):
        cit  = (r.get('citation') or '').strip()
        text = (r.get('text')     or '').strip()
        if cit and text:
            laws_lookup[cit] = text

# Prefix index: strip Abs./Ziff. so "Art. 975 ZGB" matches "Art. 975 Abs. 1 ZGB"
prefix_index: dict[str, str] = {}
for key in laws_lookup:
    base = _re.sub(r'\s+Abs\.\s+\d+\w*', '', key).strip()
    base = _re.sub(r'\s+Ziff\.\s+\d+\w*', '', base).strip()
    if base not in prefix_index:
        prefix_index[base] = key  # first matching key wins

print(f'Laws loaded       : {len(laws_lookup):,} statute entries')
print(f'Prefix index size : {len(prefix_index):,} base articles')

# Coverage check
covered = 0
for _, row in train_df.iterrows():
    golds = [c.strip() for c in str(row['gold_citations']).split(';') if c.strip()]
    if any(g in laws_lookup or g in prefix_index for g in golds):
        covered += 1
print(f'Examples with >=1 positive: {covered}/{len(train_df)}')

Laws loaded       : 175,933 statute entries
Prefix index size : 71,284 base articles
Examples with >=1 positive: 1120/1139


## 4. Build Training Triplets

For each training example:
- **Anchor**: the German legal query (prefixed with `"query: "`)
- **Positive**: actual statutory text from `laws_de.csv` for the first matched gold citation (prefixed with `"passage: "`)
- **Hard Negative**: statute text from a *different legal domain* — teaches the model that domain vocabulary alone is not relevance

Direct supervision: query → statute text. No indirect court-chunk detour.

In [5]:
def primary_domain(statutes: list[str]) -> str:
    """Return the most frequent law code among the gold statutes."""
    codes = []
    for s in statutes:
        m = re.search(r'\b([A-Z]\w+)$', s.strip())
        if m:
            codes.append(m.group(1))
    return Counter(codes).most_common(1)[0][0] if codes else 'UNKNOWN'


def find_law_text(gold_citations: list[str]) -> str | None:
    """Return statute text for the first gold citation found.
    Tries exact match first, then prefix match (strips Abs./Ziff.)."""
    for cit in gold_citations:
        text = laws_lookup.get(cit)
        if text:
            return text
    for cit in gold_citations:
        base = re.sub(r'\s+Abs\.\s+\d+\w*', '', cit).strip()
        base = re.sub(r'\s+Ziff\.\s+\d+\w*', '', base).strip()
        key  = prefix_index.get(base)
        if key:
            return laws_lookup[key]
    return None


# Build (anchor, positive, domain, gold_set) for every training example
raw_pairs: list[tuple[str, str, str, set]] = []
no_positive = 0

for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc='Building positives'):
    query  = str(row['query']).strip()
    golds  = [c.strip() for c in str(row['gold_citations']).split(';') if c.strip()]
    domain = primary_domain(golds)

    positive = find_law_text(golds)
    if positive is None:
        no_positive += 1
        continue

    raw_pairs.append(('query: ' + query, 'passage: ' + positive, domain, set(golds)))

print(f'Pairs with positive found : {len(raw_pairs)}')
print(f'Skipped (no text found)   : {no_positive}')

Building positives:   0%|          | 0/1139 [00:00<?, ?it/s]

Pairs with positive found : 1120
Skipped (no text found)   : 19


In [6]:
!pip install sentence_transformers

In [7]:
# Build domain → list of statute texts (from laws_lookup, not court chunks)
domain_pool: dict[str, list[str]] = defaultdict(list)
for _, positive, domain, _ in raw_pairs:
    domain_pool[domain].append(positive)

print('Domain distribution:')
for domain, chunks in sorted(domain_pool.items(), key=lambda x: -len(x[1])):
    print(f'  {domain:10s}: {len(chunks):4d} examples')

random.seed(42)
all_domains = list(domain_pool.keys())

training_examples: list[InputExample] = []
with_hard_neg = 0

for query, positive, domain, gold_set in raw_pairs:
    # Hard negative: statute text from a different legal domain
    other_domains = [d for d in all_domains if d != domain and domain_pool[d]]
    if other_domains:
        neg_domain    = random.choice(other_domains)
        hard_negative = random.choice(domain_pool[neg_domain])
        training_examples.append(InputExample(texts=[query, positive, hard_negative]))
        with_hard_neg += 1
    else:
        training_examples.append(InputExample(texts=[query, positive]))

print(f'\nTotal training examples : {len(training_examples)}')
print(f'  with hard negative    : {with_hard_neg}')
print(f'  pair only (fallback)  : {len(training_examples) - with_hard_neg}')

Domain distribution:
  ZGB       :  150 examples
  OR        :  119 examples
  StGB      :   77 examples
  BV        :   76 examples
  DBG       :   67 examples
  ZPO       :   47 examples
  IPRG      :   40 examples
  StPO      :   26 examples
  AIG       :   24 examples
  URG       :   23 examples
  JStPO     :   23 examples
  VwVG      :   22 examples
  SchKG     :   21 examples
  ATSG      :   21 examples
  RPG       :   20 examples
  BGG       :   19 examples
  BankG     :   18 examples
  AsylG     :   17 examples
  JStG      :   17 examples
  SVG       :   17 examples
  FINMAG    :   15 examples
  VStG      :   15 examples
  DSG       :   14 examples
  USG       :   12 examples
  FIDLEG    :   11 examples
  FinfraG   :   10 examples
  BVG       :   10 examples
  KVG       :   10 examples
  UVG       :    9 examples
  GBV       :    9 examples
  NHG       :    7 examples
  BankV     :    7 examples
  BetmG     :    7 examples
  BüG       :    7 examples
  UWG       :    6 examples

In [8]:
# Inspect a sample triplet
idx = 10
ex  = training_examples[idx]
print(f'=== Sample triplet (index {idx}) ===')
print(f'ANCHOR   : {ex.texts[0][:200]}')
print()
print(f'POSITIVE : {ex.texts[1][:200]}')
print()
if len(ex.texts) > 2:
    print(f'NEGATIVE : {ex.texts[2][:200]}')

=== Sample triplet (index 10) ===
ANCHOR   : query: Welche rechtlichen Schwierigkeiten stellen sich, wenn dieser Durchlässigkeitsgedanke nunmehr
auch in den Strafvollzug vordringt?

POSITIVE : passage: 1 Sind bei einem Verurteilten vor oder während des Vollzuges einer Freiheitsstrafe oder einer Verwahrung nach Artikel 64 Absatz 1 die Voraussetzungen einer stationären therapeutischen Massnah

NEGATIVE : passage: 2 Genehmigungsbehörde ist:a. das Inspektorat;
b. das BFE für Anlagen, bei denen das Inspektorat Einsprachen nicht erledigen oder Differenzen mit den beteiligten Bundesbehörden nicht ausräumen


## 5. Fine-Tune with MNRL

`MultipleNegativesRankingLoss` treats every other positive in the batch as an additional negative (in-batch negatives), plus the explicit hard negative from the triplet. With batch size 16 you get 15 free in-batch negatives per step.

**Expected time: 30–60 minutes on RTX 4050 (6 GB VRAM).**

> **Important**: If Mistral-7B is loaded in another kernel, unload it first to free VRAM before running this cell.

In [9]:

# Allows PyTorch to reuse fragmented VRAM blocks instead of failing on large allocations
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Free any model objects left over from a previous run in this session
for _var in ['model', 'ft_model', 'base_model']:
    if _var in globals():
        del globals()[_var]
gc.collect()

if device == 'cuda':
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM : {free:.2f} GB free / {total:.2f} GB total')
    if free < 1.5:
        print('\nWARNING: Another process is holding GPU memory. Shut it down first.')
        raise RuntimeError('Insufficient VRAM — free up GPU memory before continuing.')

model = SentenceTransformer('intfloat/multilingual-e5-base', device=device)
# 256 tokens covers full German train queries (~200-400 tokens) without the
# O(seq²) memory cost of 512. Matches deployment seq_length for consistency.
model.max_seq_length = 256

total_params = sum(p.numel() for p in model.parameters())
print(f'Model device     : {next(model.parameters()).device}')  # must say cuda:0
print(f'Model parameters : {total_params / 1e6:.0f}M')
print(f'Max seq length   : {model.max_seq_length}')
if device == 'cuda':
    print(f'VRAM after load  : {torch.cuda.memory_allocated() / 1e9:.2f} GB')

VRAM : 6.44 GB free / 6.44 GB total


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model device     : cuda:0
Model parameters : 278M
Max seq length   : 256
VRAM after load  : 1.11 GB


In [11]:
BATCH_SIZE  = 8    # physical batch per GPU step (safe for 6 GB)
GRAD_ACCUM  = 2    # accumulate 2 steps → effective batch = 16
EPOCHS      = 2
WARMUP      = 100
LR          = 2e-5

# Convert InputExample list → HuggingFace Dataset (required by sentence-transformers v3+ trainer)
triplets = [
    (ex.texts[0], ex.texts[1], ex.texts[2])
    for ex in training_examples
    if len(ex.texts) > 2
]
print(f'Triplets available  : {len(triplets)}')

train_dataset = HFDataset.from_dict({
    'anchor':   [t[0] for t in triplets],
    'positive': [t[1] for t in triplets],
    'negative': [t[2] for t in triplets],
})

train_loss = MultipleNegativesRankingLoss(model=model)

steps_per_epoch = len(triplets) // BATCH_SIZE
print(f'Batch size          : {BATCH_SIZE}  (effective {BATCH_SIZE * GRAD_ACCUM} with grad accum)')
print(f'Grad accum steps    : {GRAD_ACCUM}')
print(f'Steps per epoch     : {steps_per_epoch}')
print(f'Total update steps  : {steps_per_epoch * EPOCHS // GRAD_ACCUM}')
print(f'Warmup steps        : {WARMUP}')
print(f'Learning rate       : {LR}')
print(f'fp16 (GPU AMP)      : {device == "cuda"}')
print(f'Output path         : {MODEL_OUT}')

Triplets available  : 1120
Batch size          : 8  (effective 16 with grad accum)
Grad accum steps    : 2
Steps per epoch     : 140
Total update steps  : 140
Warmup steps        : 100
Learning rate       : 2e-05
fp16 (GPU AMP)      : True
Output path         : c:\Users\david\Desktop\project_ssl\Omnilex-Agentic-Retrieval-Competition\models\e5-base-swiss-legal-tuned


In [12]:
!pip install "accelerate>=1.1.0"

In [13]:
training_args = SentenceTransformerTrainingArguments(
    output_dir=str(MODEL_OUT),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=WARMUP,
    fp16=device == 'cuda',              # FP16 mixed precision on GPU
    bf16=False,
    gradient_checkpointing=True,        # recompute activations on backward — trades speed for VRAM
    dataloader_drop_last=True,
    dataloader_num_workers=0,           # 0 = main process only (avoids Windows fork issues)
    logging_steps=10,
    save_strategy='epoch',
    report_to='none',
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    loss=train_loss,
)

if device == 'cuda':
    torch.cuda.empty_cache()
    print(f'VRAM before training : {torch.cuda.memory_allocated() / 1e9:.2f} GB')

print('Starting training on', next(model.parameters()).device)
trainer.train()

model.save_pretrained(str(MODEL_OUT))
print(f'\nModel saved to {MODEL_OUT}')

if device == 'cuda':
    print(f'VRAM after training  : {torch.cuda.memory_allocated() / 1e9:.2f} GB')

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


VRAM before training : 1.11 GB
Starting training on cuda:0


Step,Training Loss
10,2.379674
20,2.277991
30,2.101219
40,1.845784
50,1.582630
60,1.217522
70,0.967813
80,0.960640
90,0.952116
100,0.790222


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to c:\Users\david\Desktop\project_ssl\Omnilex-Agentic-Retrieval-Competition\models\e5-base-swiss-legal-tuned
VRAM after training  : 3.36 GB


## 6. Quick Verification

Encode one inheritance-law query and compare cosine similarity to a correct civil chunk vs a criminal detention chunk. After fine-tuning the civil chunk should score higher.

In [14]:
import numpy as np

ft_model   = SentenceTransformer(str(MODEL_OUT), device=device)
base_model = SentenceTransformer('intfloat/multilingual-e5-base', device=device)
ft_model.max_seq_length   = 256
base_model.max_seq_length = 256

# Test with a German query (matching training language) against a correct statute vs wrong domain
test_query = 'query: Welche Voraussetzungen müssen für die Anordnung von Untersuchungshaft erfüllt sein?'

correct_chunk = (
    'passage: Art. 221 Abs. 1 StPO — Untersuchungs- und Sicherheitshaft sind nur zulässig, '
    'wenn die beschuldigte Person eines Verbrechens oder Vergehens dringend verdächtig ist und '
    'Fluchtgefahr, Kollusionsgefahr oder Wiederholungsgefahr besteht.'
)

wrong_chunk = (
    'passage: Art. 975 Abs. 1 ZGB — Wer durch einen unrichtigen Eintrag im Grundbuch in seinen '
    'dinglichen Rechten verletzt ist, kann auf Löschung oder Abänderung des Eintrags klagen.'
)

def cos_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

for name, m in [('BASE model (before)', base_model), ('FINE-TUNED (after)', ft_model)]:
    q_vec = m.encode([test_query],    normalize_embeddings=True)[0]
    c_vec = m.encode([correct_chunk], normalize_embeddings=True)[0]
    w_vec = m.encode([wrong_chunk],   normalize_embeddings=True)[0]
    print(f'\n{name}')
    print(f'  query ↔ StPO detention chunk : {cos_sim(q_vec, c_vec):.4f}')
    print(f'  query ↔ ZGB property chunk   : {cos_sim(q_vec, w_vec):.4f}')
    print(f'  gap (correct - wrong)        : {cos_sim(q_vec, c_vec) - cos_sim(q_vec, w_vec):+.4f}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



BASE model (before)
  query ↔ StPO detention chunk : 0.8760
  query ↔ ZGB property chunk   : 0.7865
  gap (correct - wrong)        : +0.0895

FINE-TUNED (after)
  query ↔ StPO detention chunk : 0.7978
  query ↔ ZGB property chunk   : 0.2379
  gap (correct - wrong)        : +0.5598


## 7. Re-Index LanceDB\n\nThe 2.5M vectors in the existing `lancedb_courts` table were produced by the old model. They must be regenerated.\n\n**Expected time: ~1 hour on RTX 4050 with clean VRAM.**\n\n> **Run this section in isolation — restart the kernel first, then run only:**\n> 1. **Cell 1** (imports + paths + device)\n> 2. **This section** (cells below)\n>\n> Do NOT re-run any training cells first. Training leaves optimizer states in VRAM\n> that eat 4+ GB, reducing encoding speed from ~1s/it to ~65s/it.

In [2]:
import lancedb
import numpy as np
import pyarrow as pa
import time as _time

COURTS_LANCE_PATH = INDEX_PATH / 'lancedb_courts_finetuned'
LAWS_LANCE_PATH   = INDEX_PATH / 'lancedb_laws_finetuned'

ft_model = SentenceTransformer(str(MODEL_OUT), device=device)
ft_model.max_seq_length = 256

if device == 'cuda':
    ft_model.half()  # Casts weights to 16-bit floats

print(f'ft_model device : {next(ft_model.parameters()).device}')
if device == 'cuda':
    print(f'VRAM used       : {torch.cuda.memory_allocated() / 1e9:.2f} GB')

_t0 = _time.time()
_test = ft_model.encode(
    ['passage: ' + 'test sentence ' * 10] * 512,
    batch_size=512, normalize_embeddings=True, show_progress_bar=False,
)
_elapsed = _time.time() - _t0
print(f'Speed check: 512 sentences in {_elapsed:.2f}s  →  {512/_elapsed:.0f} sent/s')
if _elapsed > 30:
    raise RuntimeError('Too slow — model is likely on CPU. Check device assignment.')

# ── Zero-copy flush: numpy → pyarrow (no Python float objects) ───────────────
VEC_DIM = 768

def _to_arrow(cits, texts, vecs_list):
    mat      = np.vstack(vecs_list).astype(np.float32)
    flat_buf = pa.array(mat.ravel(), type=pa.float32())
    return pa.table({
        'citation': pa.array(cits,  type=pa.string()),
        'text':     pa.array(texts, type=pa.string()),
        'vector':   pa.FixedSizeListArray.from_arrays(flat_buf, VEC_DIM),
    })

# ── Part A: Re-index courts (2.5M rows) ──────────────────────────────────────
# Read the entire CSV into RAM once — avoids slow incremental CSV parsing per chunk
print('\nLoading court CSV into RAM ...')
_t1 = _time.time()
_courts_df = pd.read_csv(
    DATA_PATH / 'court_considerations.csv',
    usecols=['citation', 'text'], dtype=str,
).dropna(subset=['citation', 'text'])
all_cits  = _courts_df['citation'].tolist()
all_texts = _courts_df['text'].tolist()
del _courts_df
print(f'Loaded {len(all_cits):,} rows in {_time.time()-_t1:.0f}s')

ENCODE_BATCH = 4096
WRITE_BATCH  = 100_000

db_courts  = lancedb.connect(str(COURTS_LANCE_PATH))
tbl_courts = None
rows_done  = 0
buf_cits, buf_texts, buf_vecs = [], [], []

def flush_courts(cits, texts, vecs_list, tbl):
    arrow_tbl = _to_arrow(cits, texts, vecs_list)
    if tbl is None:
        return db_courts.create_table('courts', data=arrow_tbl, mode='overwrite')
    tbl.add(arrow_tbl)
    return tbl

print('Encoding + indexing courts ...')
for i in tqdm(range(0, len(all_cits), ENCODE_BATCH), desc='Courts'):
    cits  = all_cits[i : i + ENCODE_BATCH]
    texts = all_texts[i : i + ENCODE_BATCH]
    vecs  = ft_model.encode(
        ['passage: ' + t for t in texts],
        batch_size=256, normalize_embeddings=True, show_progress_bar=False,
    )  # numpy (N, 768) — no .tolist()
    buf_cits.extend(cits); buf_texts.extend(texts); buf_vecs.append(vecs)
    if len(buf_cits) >= WRITE_BATCH:
        tbl_courts = flush_courts(buf_cits, buf_texts, buf_vecs, tbl_courts)
        rows_done += len(buf_cits)
        buf_cits.clear(); buf_texts.clear(); buf_vecs.clear()

if buf_cits:
    tbl_courts = flush_courts(buf_cits, buf_texts, buf_vecs, tbl_courts)
    rows_done += len(buf_cits)
del all_cits, all_texts
print(f'Courts done: {rows_done:,} vectors → {COURTS_LANCE_PATH}')

# ── Part B: Index laws (175K rows) ───────────────────────────────────────────
print('\nLoading laws CSV into RAM ...')
_laws_df  = pd.read_csv(
    DATA_PATH / 'laws_de.csv',
    usecols=['citation', 'text'], dtype=str,
).dropna(subset=['citation', 'text'])
law_cits  = _laws_df['citation'].tolist()
law_texts = _laws_df['text'].tolist()
del _laws_df
print(f'Loaded {len(law_cits):,} rows')

db_laws  = lancedb.connect(str(LAWS_LANCE_PATH))
tbl_laws = None
law_rows = 0
buf_cits, buf_texts, buf_vecs = [], [], []

def flush_laws(cits, texts, vecs_list, tbl):
    arrow_tbl = _to_arrow(cits, texts, vecs_list)
    if tbl is None:
        return db_laws.create_table('laws', data=arrow_tbl, mode='overwrite')
    tbl.add(arrow_tbl)
    return tbl

print('Encoding + indexing laws ...')
for i in tqdm(range(0, len(law_cits), ENCODE_BATCH), desc='Laws'):
    cits  = law_cits[i : i + ENCODE_BATCH]
    texts = law_texts[i : i + ENCODE_BATCH]
    vecs  = ft_model.encode(
        ['passage: ' + t for t in texts],
        batch_size=256, normalize_embeddings=True, show_progress_bar=False,
    )
    buf_cits.extend(cits); buf_texts.extend(texts); buf_vecs.append(vecs)
    if len(buf_cits) >= WRITE_BATCH:
        tbl_laws = flush_laws(buf_cits, buf_texts, buf_vecs, tbl_laws)
        law_rows += len(buf_cits)
        buf_cits.clear(); buf_texts.clear(); buf_vecs.clear()

if buf_cits:
    tbl_laws = flush_laws(buf_cits, buf_texts, buf_vecs, tbl_laws)
    law_rows += len(buf_cits)

print(f'Laws done: {law_rows:,} vectors → {LAWS_LANCE_PATH}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ft_model device : cuda:0
VRAM used       : 0.57 GB
Speed check: 512 sentences in 1.01s  →  509 sent/s

Loading court CSV into RAM ...
Loaded 2,476,315 rows in 33s
Encoding + indexing courts ...


Courts:   0%|          | 0/605 [00:00<?, ?it/s]

Courts done: 2,476,315 vectors → c:\Users\david\Desktop\project_ssl\Omnilex-Agentic-Retrieval-Competition\data\processed\lancedb_courts_finetuned

Loading laws CSV into RAM ...
Loaded 175,933 rows
Encoding + indexing laws ...


Laws:   0%|          | 0/43 [00:00<?, ?it/s]

Laws done: 175,933 vectors → c:\Users\david\Desktop\project_ssl\Omnilex-Agentic-Retrieval-Competition\data\processed\lancedb_laws_finetuned


In [3]:
# Build IVF-PQ ANN indexes for both tables
print('Building ANN index for courts ...')
tbl_courts.create_index(metric='cosine', num_partitions=256, num_sub_vectors=96)
print(f'Courts index built. Rows: {tbl_courts.count_rows():,}')

print('Building ANN index for laws ...')
tbl_laws.create_index(metric='cosine', num_partitions=64, num_sub_vectors=96)
print(f'Laws index built. Rows: {tbl_laws.count_rows():,}')

Building ANN index for courts ...
Courts index built. Rows: 2,476,315
Building ANN index for laws ...
Laws index built. Rows: 175,933


## 8. Update the Pipeline Notebook

Once re-indexing is complete, update `graphrag.ipynb` cell 9 with the fine-tuned model and both new LanceDB paths:

```python
# Embedding model — fine-tuned on Swiss legal statute text
_embed_model = SentenceTransformer(str(REPO_ROOT / "models" / "e5-base-swiss-legal-tuned"), device=_device)
_embed_model.max_seq_length = 256

# Courts dense index (fine-tuned vectors)
LANCEDB_COURTS_PATH = REPO_ROOT / "data" / "processed" / "lancedb_courts_finetuned"
_lance_db    = lancedb.connect(str(LANCEDB_COURTS_PATH))
_lance_table = _lance_db.open_table("courts")

# Laws dense index (new — fine-tuned vectors over laws_de.csv)
LANCEDB_LAWS_PATH  = REPO_ROOT / "data" / "processed" / "lancedb_laws_finetuned"
_lance_db_laws     = lancedb.connect(str(LANCEDB_LAWS_PATH))
_lance_table_laws  = _lance_db_laws.open_table("laws")
```

Then in `run_dag_pipeline`, pass `lance_table_laws=_lance_table_laws` and add a HyDE dense search over the laws table in parallel to the existing courts dense search.